# Introdução ao Sionna RT

O ray tracing é uma técnica para simular realizações de canais específicas ao ambiente e fisicamente precisas para um determinado cenário e posição do usuário.
Consulte o [EM Primer](https://nvlabs.github.io/sionna/rt/em_primer.html) para obter mais detalhes sobre os fundamentos teóricos do ray tracing de canais sem fio.

O Sionna RT é uma biblioteca de ray-tracing, acelerada por hardware e de código aberto, destinada à modelagem da propagação de ondas de rádio, desenvolvido com base no [Mitsuba 3](https://www.mitsuba-renderer.org/). O Mitsuba 3 é um sistema de renderização para simulação que utiliza o compilador just-in-time [Dr.Jit](https://drjit.readthedocs.io/en/latest/).

Graças ao cálculo automático de gradientes do Dr. Jit, os gradientes das funções das respostas dos canais ou dos mapas de rádio em relação à maioria dos parâmetros do processo de traçado de raios — incluindo propriedades dos materiais, padrões de antenas e de dispersão, orientações e posições dos objetos — podem ser calculados com eficiência e utilizados em diversos problemas de otimização baseados em gradientes.

O Sionna RT utiliza o Mitsuba 3 para a renderização e o gerenciamento de cenas, usando por exemplo, seu formato de arquivo XML.
Ao ser importado, o Sionna RT configura automaticamente a [variante do Mitsuba](https://mitsuba.readthedocs.io/en/stable/src/key_topics/variants.html) correta com base no hardware disponível.


Os arquivos de cena para o Mitsuba 3 podem ser criados, editados e exportados com o popular pacote de criação 3D de código aberto [Blender](https://www.blender.org/) e o [complemento Mitsuba-Blender](https://github.com/mitsuba-renderer/mitsuba-blender). É possível criar cenas rapidamente a partir de praticamente qualquer lugar do mundo usando o [OpenStreetMap](https://www.openstreetmap.org/) e o [plug-in Blender-OSM](https://prochitecture.gumroad.com/l/blender-osm). No Sionna, os cenário e os paths da onda de rádio podem ser renderizados através das lentes de câmeras configuráveis por meio de ray tracing ou exibidos com um visualizador 3D integrado. Para mais detalhes sobre a criação e renderização de cenas, consulte a [documentação da API do Sionna RT](https://nvlabs.github.io/sionna/rt/api/rt.html) e o [tutorial em vídeo](https://youtu.be/7xHLDxUaQ7c) disponível.

## Imports

In [ ]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    # Only auto-install when the package itself is missing, not when a
    # transitive dependency fails to import.
    if getattr(e, "name", None) not in ("sionna", "sionna.rt"):
        raise
    import os
    import sys
    if "google.colab" in sys.modules:
        # Install Sionna RT in Google Colab
        print("Installing Sionna RT and restarting the runtime. Please run the cell again.")
        os.system("pip install sionna-rt")
        os.kill(os.getpid(), 5)
    else:
        raise e

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import mitsuba as mi
import numpy as np

no_preview = True # Toggle to False to use the preview widget

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies

# Query the Mitsuba variant that was automatically selected by Sionna RT.
# There should be no need to adjust it, but if you do, make sure to select
# a variant ending in `*_ad_mono_polarized`.
print(f"Mitsuba variant: {mi.variant()}")

## Carregamento e visualização de cenas

O Sionna RT pode carregar arquivos de cena externos (no formato de arquivo XML do Mitsuba) ou uma das [cenas integradas](https://nvlabs.github.io/sionna/rt/api/scene.html#examples).

Neste exemplo, carregamos uma cena de exemplo que contém a área ao redor da Frauenkirche, em Munique, na Alemanha.

In [ ]:
# Load integrated scene
scene = load_scene(sionna.rt.scene.munich) # Try also sionna.rt.scene.etoile

Para visualizar a cena, podemos usar a função `preview`, que abre uma pré-visualização interativa da cena.
Isso só funciona em notebooks do Jupyter.

Você pode usar os seguintes controles:

- Botão esquerdo do mouse: girar
- Roda de rolagem: ampliar/reduzir
- Botão direito do mouse: mover

Observe que apenas uma instância de pré-visualização por cena pode ser aberta ao mesmo tempo.
No entanto, várias cenas podem ser carregadas em paralelo.

In [ ]:
if not no_preview:
    scene.preview();

Muitas vezes, é conveniente escolher um ponto de vista na visualização 3D antes de renderizá-la como uma imagem de alta qualidade.
A célula a seguir utiliza a câmera “preview”, que corresponde ao ponto de vista da imagem de visualização atual.

In [ ]:
# Only availabe if a preview is open
if not no_preview:
    scene.render(camera="preview", num_samples=512);

Também é possível salvar a imagem em um arquivo, conforme mostrado a seguir:

In [ ]:
# Only availabe if a preview is open
if not no_preview:
    scene.render_to_file(camera="preview",
                         filename="scene.png",
                         resolution=[650,500]);

Em vez da câmera de visualização, também é possível especificar câmeras dedicadas com posições e direções de visualização diferentes.

In [ ]:
# Create new camera with different configuration
my_cam = Camera(position=[-250,250,100], look_at=[-15,30,28])

# Render scene with new camera*
scene.render(camera=my_cam, resolution=[650, 500], num_samples=512); # Increase num_samples to increase image quality

## Inspeção de SceneObjects e edição de cenas

Uma cena é composta por vários [SceneObjects](https://nvlabs.github.io/sionna/rt/api/scene_object.html), que podem ser acessados da seguinte maneira:

In [ ]:
scene = load_scene(sionna.rt.scene.simple_street_canyon, merge_shapes=False)
scene.objects

In [ ]:
floor = scene.get("floor")

Os SceneObjects podem ser transformados por meio das seguintes propriedades e métodos: 
- posição
- orientação
- escala

In [ ]:
print("Position (x,y,z) [m]: ", floor.position)
print("Orientation (alpha, beta, gamma) [rad]: ", floor.orientation)
print("Scaling: ", floor.scaling)

Mais detalhes sobre essas funcionalidades podem ser encontrados no [Tutorial sobre carregamento e edição de cenas](https://nvlabs.github.io/sionna/rt/tutorials/Scene-Edit.html).

Todo SceneObject possui outra propriedade importante, o vetor `velocity`:

In [ ]:
print("Velocity (x,y,z) [m/s]: ", floor.velocity)

Essa propriedade é utilizada durante o processo de ray tracing para calcular um desvio Doppler para cada caminho de propagação. Essas informações podem, então, ser utilizadas para calcular sinteticamente a evolução temporal das respostas impulsionais dos canais. Mais detalhes sobre esse assunto são fornecidos no [Tutorial sobre Mobilidade](https://nvlabs.github.io/sionna/rt/tutorials/Mobility.html).

A última propriedade dos SceneObjects que abordaremos aqui é o [RadioMaterial](https://nvlabs.github.io/sionna/rt/api/radio_materials.html):

In [ ]:
floor.radio_material

O material de rádio determina como um objeto interage com as ondas de rádio incidentes. Para saber mais sobre materiais de rádio e como eles podem ser modificados, convidamos você a dar uma olhada no Guia do Desenvolvedor sobre [Noções básicas sobre materiais de rádio](https://nvlabs.github.io/sionna/rt/developer/dev_custom_radio_materials.html).

Dependendo do tipo de material de rádio, algumas de suas propriedades podem mudar em função da frequência da onda de rádio incidente:

In [ ]:
scene.frequency = 28e9 # in Hz; implicitly updates RadioMaterials that implement frequency dependent properties
floor.radio_material # Note that the conductivity (sigma) changes automatically

## Ray-tracing das trajetórias de propagação

Assim que uma cena é carregada, podemos posicionar transmissores e receptores nela e calcular as trajetórias de propagação entre eles. Todos os transmissores e todos os receptores estão equipados com os conjuntos de antenas, definidos pelas propriedades `scene.tx_array` e `scene.rx_array` da `scene`, respectivamente. Os conjuntos de antenas são compostos por várias antenas idênticas. As antenas podem ter diagramas de radiação personalizados ou predefinidos e podem ser de polarização única ou dupla. Uma cena pode conter vários transmissores e receptores, desde que tenham nomes exclusivos.

Mais detalhes sobre os diagramas de antena podem ser encontrados no Guia do Desenvolvedor [Noções básicas sobre rádio](https://nvlabs.github.io/sionna/rt/developer/dev_custom_radio_materials.html).

In [ ]:
scene = load_scene(sionna.rt.scene.munich, merge_shapes=True) # Merge shapes to speed-up computations

# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="cross")

# Create transmitter
tx = Transmitter(name="tx",
                 position=[8.5,21,27],
                 display_radius=2)

# Add transmitter instance to scene
scene.add(tx)

# Create a receiver
rx = Receiver(name="rx",
              position=[45,90,1.5],
              display_radius=2)

# Add receiver instance to scene
scene.add(rx)

tx.look_at(rx) # Transmitter points towards receiver

Os caminhos de propagação são calculados com a ajuda de um [PathSolver](https://nvlabs.github.io/sionna/rt/api/paths_solvers.html).
A célula a seguir mostra como esse solucionador de trajetórias é instanciado e utilizado. 

O parâmetro `max_depth` determina o número máximo de interações entre um raio e os objetos da cena. 
Por exemplo, com um `max_depth` igual a zero, apenas os caminhos de linha de visada (LoS) são considerados. Para um `max_depth` igual a um, são considerados tanto os caminhos de linha de visada quanto os caminhos que tenham sofrido apenas uma reflexão e refração antes de chegarem no receptor. Quando o argumento `synthetic_array` é definido como `False`, os arranjos de antenas são modelados explicitamente, localizando-se caminhos entre qualquer par de antenas transmissoras e receptoras na cena. Caso contrário, os arranjos de antenas são representados por uma única antena localizada no centro do arranjo.

Os deslocamentos de fase relacionados às posições relativas das antenas serão então aplicados com base na suposição de onda plana ao se calcularem as respostas impulsionais do canal.

In [ ]:
# Instantiate a path solver
# The same path solver can be used with multiple scenes
p_solver  = PathSolver()

# Compute propagation paths
paths = p_solver(scene=scene,
                 max_depth=5,
                 los=True,
                 specular_reflection=True,
                 diffuse_reflection=False,
                 refraction=True,
                 synthetic_array=False,
                 seed=41)

O objeto [Paths](https://nvlabs.github.io/sionna/rt/paths.html) contém todos os caminhos que foram encontrados entre transmissores e receptores.
Em princípio, a existência de cada caminho é determinística para uma determinada posição e ambiente. Por padrão, no entanto, o `PathSolver` não garante resultados determinísticos, mesmo quando `seed` é fixo. Use `PathSolver(deterministic=True)` quando forem necessários resultados determinísticos.

Vamos agora visualizar os caminhos encontrados na cena:

In [ ]:
if no_preview:
    scene.render(camera=my_cam, paths=paths, clip_at=20);
else:
    scene.preview(paths=paths, clip_at=20);

O objeto Paths contém informações detalhadas sobre cada caminho encontrado e nos permite gerar respostas de impulso de canal e aplicar deslocamentos Doppler para a simulação da evolução temporal. Para uma descrição detalhada, consulte o guia do desenvolvedor [Entendendo o objeto Paths](https://nvlabs.github.io/sionna/rt/developer/dev_understanding_paths.html).

## Obtendos as respostas impulsionais e de frequência do canal a partir das trajetórias

Uma vez calculadas as trajetórias, elas podem ser transformadas em uma resposta impulsional do canal (CIR) equivalente à banda base por meio de [Paths.cir()](https://nvlabs.github.io/sionna/rt/api/paths.html#sionna.rt.Paths.cir), em uma resposta ao impulso discreta, complexa e equivalente à banda base por meio de [Paths.taps()](https://nvlabs.github.io/sionna/rt/api/paths.html#sionna.rt.Paths.taps), ou em uma resposta em frequência do canal (CFR) por meio de [Paths.cfr()](https://nvlabs.github.io/sionna/rt/api/paths.html#sionna.rt.Paths.cfr). Esses métodos de classe podem simular a evolução temporal do canal com base nos deslocamentos Doppler calculados ([Paths.doppler](https://nvlabs.github.io/sionna/rt/api/paths.html#sionna.rt.Paths.doppler)).

Vamos primeiro derivar e visualizar a resposta impulsional do canal equivalente à banda base a partir dos caminhos calculados acima:

In [ ]:
a, tau = paths.cir(normalize_delays=True, out_type="numpy")

# Shape: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
print("Shape of a: ", a.shape)

# Shape: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths]
print("Shape of tau: ", tau.shape)

O argumento `out_type` pode ser usado para converter o CIR em tensores de diferentes frameworks, como [Dr.Jit](https://drjit.readthedocs.io/en/latest/reference.html) (“drjit”), [Numpy](https://numpy.org) (“numpy”),
            [Jax](https://jax.readthedocs.io/en/latest/index.html) (“jax”),
            [TensorFlow](https://www.tensorflow.org) (“tf”),
            e [PyTorch](https://pytorch.org) (“torch”). Consulte o guia do desenvolvedor [Compatibilidade com outros frameworks](https://nvlabs.github.io/sionna/rt/developer/dev_compat_frameworks.html) para obter mais informações sobre a interoperabilidade do Sionna RT com outros frameworks, incluindo a propagação de gradientes.

In [ ]:
t = tau[0,0,0,0,:]/1e-9 # Scale to ns
a_abs = np.abs(a)[0,0,0,0,:,0]
a_max = np.max(a_abs)

# And plot the CIR
plt.figure()
plt.title("Channel impulse response")
plt.stem(t, a_abs)
plt.xlabel(r"$\tau$ [ns]")
plt.ylabel(r"$|a|$");

Observe que o atraso do primeiro caminho a chegar é, por padrão, normalizado para zero. Esse comportamento pode ser alterado definindo-se o argumento ``normalize_delays`` como `False`.

Podemos obter a resposta em frequência do canal de maneira semelhante:

In [ ]:
# OFDM system parameters
num_subcarriers = 1024
subcarrier_spacing=30e3

# Compute frequencies of subcarriers relative to the carrier frequency
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)

# Compute channel frequency response
h_freq = paths.cfr(frequencies=frequencies,
                   normalize=True, # Normalize energy
                   normalize_delays=True,
                   out_type="numpy")

# Shape: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, num_subcarriers]
print("Shape of h_freq: ", h_freq.shape)

# Plot absolute value
plt.figure()
plt.plot(np.abs(h_freq)[0,0,0,0,0,:]);
plt.xlabel("Subcarrier index");
plt.ylabel(r"|$h_\text{freq}$|");
plt.title("Channel frequency response");

Para simulações link-level no domínio do tempo, frequentemente precisamos da resposta impulsional do canal discreta de banda base ou, simplesmente, dos taps do canal.
Estes são obtidos por meio da amostragem da resposta impulsional do canal, idealmente filtrada por um filtro passa-baixa, na frequência de amostragem desejada. Por padrão, presume-se que a amostragem seja realizada na taxa de Nyquist.

Como o filtro sinc subjacente tem uma resposta infinitamente longa, os taps do canal precisam ser truncados em um valor mínimo e máximo, ou seja, `l_min` e `l_max`, respectivamente.

In [ ]:
taps = paths.taps(bandwidth=100e6, # Bandwidth to which the channel is low-pass filtered
                  l_min=-6,        # Smallest time lag
                  l_max=100,       # Largest time lag
                  sampling_frequency=None, # Sampling at Nyquist rate, i.e., 1/bandwidth
                  normalize=True,  # Normalize energy
                  normalize_delays=True,
                  out_type="numpy")
print("Shape of taps: ", taps.shape)

plt.figure()
plt.stem(np.arange(-6, 101), np.abs(taps)[0,0,0,0,0]);
plt.xlabel(r"Tap index $\ell$");
plt.ylabel(r"|$h[\ell]|$");
plt.title("Discrete channel taps");

Cada dispositivo de rádio e objeto da cena possui um vetor de velocidade associado. Esses vetores são utilizados para calcular os deslocamentos Doppler específicos de cada trajeto, o que permite a simulação da mobilidade. Mais detalhes podem ser encontrados no [Tutorial sobre Mobilidade](https://nvlabs.github.io/sionna/rt/tutorials/Mobility.html).

Agora, atribuiremos um vetor de velocidade diferente de zero ao transmissor, recalcularemos os caminhos de propagação e calcularemos uma resposta impulsional do canal que varia com o tempo:

In [ ]:
scene.get("tx").velocity = [10, 0, 0]

# Recompute propagation paths
paths_mob = p_solver(scene=scene,
                     max_depth=5,
                     los=True,
                     specular_reflection=True,
                     diffuse_reflection=False,
                     refraction=True,
                     synthetic_array=True,
                     seed=41)

# Compute CIR with time-evolution
num_time_steps=100
sampling_frequency = 1e4
a_mob, _ = paths_mob.cir(sampling_frequency=sampling_frequency,
                         num_time_steps=num_time_steps,
                         out_type="numpy")

# Inspect time-evolution of a single path coefficient
plt.figure()
plt.plot(np.arange(num_time_steps)/sampling_frequency*1000,
         a_mob[0,0,0,0,0].real);
plt.xlabel("Time [ms]");
plt.ylabel(r"$\Re\{a_0(t) \}$");
plt.title("Time-evolution of a path coefficient");

## Mapas de rádio

O Sionna RT pode calcular mapas de rádio para todos os transmissores em uma cena. Um [RadioMap](https://nvlabs.github.io/sionna/rt/api/radio_maps.html) atribui uma métrica — como ganho de trajetória, intensidade do sinal recebido (RSS) ou relação sinal-ruído-mais-interferência (SINR) — para um transmissor específico a cada ponto em um plano. Em outras palavras, para um determinado transmissor, ele associa cada ponto da região ao ganho de canal, RSS ou SINR que um receptor com uma orientação específica observaria naquele ponto.

Assim como o cálculo das trajetórias de propagação requer um [PathSolver](https://nvlabs.github.io/sionna/rt/api/paths_solvers.html), o cálculo dos mapas de rádio requer um [RadioMapSolver](https://nvlabs.github.io/sionna/rt/api/radio_map_solvers.html). O trecho de código a seguir mostra como um mapa de rádio pode ser calculado e exibido.

Mais informações sobre mapas de rádio podem ser encontradas no [Tutorial detalhado sobre mapas de rádio](https://nvlabs.github.io/sionna/rt/tutorials/Radio-Maps.html).

In [ ]:
rm_solver = RadioMapSolver()

rm = rm_solver(scene=scene, 
               max_depth=5,
               cell_size=[1,1],
               samples_per_tx=10**6)

In [ ]:
if no_preview:
    scene.render(camera=my_cam, radio_map=rm);
else:
    scene.preview(radio_map=rm);

In [ ]:
rm.show(metric="rss", tx=0)

## Resumo

Neste tutorial, você aprendeu os conceitos básicos do Sionna RT. Agora você sabe como localizar trajetórias em ambientes complexos
e como o CIR, o CFR e os taps podem ser derivados a partir delas. Você também aprendeu como criar mapas de rádio.

Há um recurso fundamental do Sionna RT que não foi abordado neste caderno: o cálculo automático de gradientes.
Assim como a maioria dos componentes do Sionna, o Sionna RT também é configurável em relação à maioria dos parâmetros, tais como materiais radioativos, padrões de espalhamento e de antenas, orientações do transmissor e do receptor, geometrias de arranjos, posições, etc.

Dê uma olhada na [documentação da API](https://nvlabs.github.io/sionna/rt/api/rt.html) dos diversos componentes e nos outros [tutoriais](https://nvlabs.github.io/sionna/rt/tutorials.html) e [guias para desenvolvedores](https://nvlabs) disponíveis. github.io/sionna/rt/developer/developer.html).